# Internship Scam Detection using Machine Learning
#### Module: Binary Classification

---

### Project Aim & Objective
The primary goal of this project is to build an end-to-end machine learning pipeline that automatically identifies and classifies fraudulent internship postings. This notebook serves as a practical, beginner-friendly guide to detecting these malicious postings using metadata analytics.

---

### Pipeline Architecture
1. **Exploratory Data Analysis (EDA)**
2. **Train-Test Separation**
3. **Data Imputation & Preprocessing**
4. **Model Training**
5. **Evaluation**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Basic data exploration

In [ ]:
import gdown
import os

file_id = '1JzvRWYKAjDbjhuVZSMJvIbGJryxyVKQo'
url = f'https://drive.google.com/uc?id={file_id}'
output_path = '../data/internship_scam_data.csv'

os.makedirs('../data', exist_ok=True)

gdown.download(url, output_path, quiet=False)

In [ ]:
df = pd.read_csv("../data/internship_scam_data.csv")
cols = ['internship_title', 'employment_type', 'work_mode', 'industry', 
        'location', 'company_name', 'company_size', 'stipend', 
        'payment_required', 'registration_fee', 'recruiter_email_type', 
        'website_available', 'is_fake_posting']

df = df[cols]
df.head()

In [ ]:
df.info()

Null columns: `stipend`: a numerical column
#### Strategy for replacing NULLS:
1. `stipend` : value 0

## EDA

In [ ]:
df.hist(bins=100, figsize=(20,14))
plt.show()

In [ ]:
df["stipend"].min()

> stipend is capped at 2000, uncapped on the upper bound. Our strategy of replacing Nan stipend with 0s persists.

---
### correlation between all numeric columns and `is_fake_posting`

In [ ]:
corr_matrix=df.corr(numeric_only=True)
corr_matrix["is_fake_posting"].sort_values(ascending=False)

In [ ]:
for col in df.select_dtypes(include='str').columns:
    print(df[col].value_counts())

In [ ]:
df.columns

### Drop not-so-useful columns
1. High Cardinality Fields: `company_name`, `internship_title`, `location`, `employment_type`, `industry`


In [ ]:
df.drop(['company_name', 'internship_title', 'location', 'employment_type', 'industry'], axis=1, inplace=True)

### Train Test Split

In [ ]:
# split into X and y
X = df.drop(columns=['is_fake_posting'])
y = df['is_fake_posting']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.columns

segregating numerical and categorical columns

In [ ]:
# Automatically collect all other clean numeric features
all_numeric = X.select_dtypes(include=[np.number]).columns.tolist()
clean_numeric_cols = [c for c in all_numeric if c not in (['stipend'])]

# Automatically collect categorical columns for encoding
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

### Automated ColumnTransformer Pipelines

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Build individual step pipelines
stipend_pipeline = Pipeline([
    ('imputer_zero', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

numeric_clean_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [ ]:
# Assemble everything into a single ColumnTransformer layout
preprocessing_pipeline = ColumnTransformer(transformers=[
    ('stipend_path', stipend_pipeline, ['stipend']),
    ('num_clean_path', numeric_clean_pipeline, clean_numeric_cols),
    ('cat_path', categorical_pipeline, categorical_cols)
])

### Baseline model: DecisionTreeClassifier

In [ ]:
# Fit and transform the training split; transform the testing split cleanly
X_train_prepared = preprocessing_pipeline.fit_transform(X_train)
X_test_prepared = preprocessing_pipeline.transform(X_test)

### GridSearchCV Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# 1. Initialize a base Decision Tree
base_tree = DecisionTreeClassifier(random_state=42)

# 2. Define the exact parameter grid you want to test
params = {
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 10, 20],
    'criterion': ['gini', 'entropy']
}

# 3. Set up the Grid Search engine
grid_search = GridSearchCV(
    estimator=base_tree, 
    param_grid=params, 
    cv=3, 
    scoring='f1_macro',
    verbose=1
)

print("Starting Grid Search across combinations...")
# 4. Run the search on your clean, prepared training data
grid_search.fit(X_train_prepared, y_train)

# 5. Print the mathematically optimal parameters discovered
print("\n--- Optimal Hyperparameters Found ---")
print(grid_search.best_params_)

# 6. Extract the best optimized model directly
best_tree_model = grid_search.best_estimator_

# 7. Predict and evaluate using the optimized model on your isolated test set
y_pred_opt = best_tree_model.predict(X_test_prepared)

print("\n--- Optimized Decision Tree Evaluation Report ---")
print(classification_report(y_test, y_pred_opt))

In [ ]:
feature_names = preprocessing_pipeline.get_feature_names_out()
importances = pd.Series(best_tree_model.feature_importances_, index=feature_names)
print(importances.sort_values(ascending=False))

### saving the model and the pipeline

In [ ]:
import joblib
joblib.dump(best_tree_model, '../models/model.pkl')
joblib.dump(preprocessing_pipeline, '../models/pipeline.pkl')

### Key Metrics:
* **Overall Accuracy: 70%**
* **Scam Precision: 74%**
* **Scam Recall: 63%**